# 02. VQE·QAOA·PDE와 선형계 타당성 실습

모든 계산은 작은 NumPy toy model이다. 양자 SDK나 실제 QPU를 사용하지 않으며 양자 우위를 재현하지 않는다. 목표는 exact/classical reference를 먼저 만들고 비교 계약과 비용 질문을 익히는 것이다.

In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)
SEED = 20260913
rng = np.random.default_rng(SEED)
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def expectation(state, operator):
    return float(np.vdot(state, operator @ state).real)

## 1. 단일 qubit VQE

Hamiltonian $H=X+Z$의 ground energy를 $R_y(\theta)|0>$ ansatz와 grid search로 찾는다. 작은 문제이므로 exact diagonalization을 정답 oracle로 사용한다.

In [ ]:
hamiltonian = X + Z
exact_values, exact_vectors = np.linalg.eigh(hamiltonian)
exact_ground_energy = float(exact_values[0])

def ry_state(theta):
    return np.array([np.cos(theta / 2), np.sin(theta / 2)], dtype=complex)

theta_grid = np.linspace(0, 2 * np.pi, 4_001)
energies = np.array([expectation(ry_state(theta), hamiltonian) for theta in theta_grid])
best_index = int(np.argmin(energies))
vqe_energy = float(energies[best_index])
vqe_theta = float(theta_grid[best_index])

assert abs(vqe_energy - exact_ground_energy) < 1e-10
print('exact ground energy:', exact_ground_energy)
print('toy VQE energy:', vqe_energy, 'theta:', vqe_theta)
print('absolute error:', abs(vqe_energy - exact_ground_energy))

실제 VQE에서는 Hamiltonian term grouping, ansatz expressibility, optimizer evaluation, shots, noise와 transpilation 비용을 별도로 기록한다. 이 grid search의 evaluation 수는 `len(theta_grid)`이며 실제 QPU 실험의 무료 계산으로 간주할 수 없다.

## 2. 3-node path MaxCut의 QAOA p=1

모든 bitstring의 cut 값을 먼저 계산하고 brute-force optimum을 만든다. QAOA 기대값, optimum sample 확률과 approximation ratio는 서로 다른 지표다.

In [ ]:
n_qubits = 3
edges = [(0, 1), (1, 2)]
dimension = 2 ** n_qubits
bitstrings = [format(index, f'0{n_qubits}b') for index in range(dimension)]
costs = np.array([sum(bits[u] != bits[v] for u, v in edges) for bits in bitstrings], dtype=float)
classical_optimum = float(costs.max())
plus_state = np.ones(dimension, dtype=complex) / np.sqrt(dimension)

def qaoa_state(gamma, beta):
    cost_evolved = np.exp(-1j * gamma * costs) * plus_state
    one_qubit_mixer = np.cos(beta) * I - 1j * np.sin(beta) * X
    mixer = np.kron(np.kron(one_qubit_mixer, one_qubit_mixer), one_qubit_mixer)
    return mixer @ cost_evolved

best = {'expectation': -np.inf}
for gamma in np.linspace(0, np.pi, 61):
    for beta in np.linspace(0, np.pi / 2, 61):
        state = qaoa_state(gamma, beta)
        probs = np.abs(state) ** 2
        value = float(probs @ costs)
        if value > best['expectation']:
            best = {'gamma': gamma, 'beta': beta, 'expectation': value, 'probs': probs}

success_probability = float(best['probs'][costs == classical_optimum].sum())
approximation_ratio = best['expectation'] / classical_optimum
assert best['expectation'] <= classical_optimum + 1e-12
assert approximation_ratio > 0.7
print('classical optimum:', classical_optimum)
print('best p=1 expected cut:', best['expectation'])
print('expectation approximation ratio:', approximation_ratio)
print('optimum-sample probability:', success_probability)

## 3. 1D Poisson 방정식의 고전 baseline

$-u''(x)=\pi^2\sin(\pi x)$, $u(0)=u(1)=0$의 정확한 해는 $u(x)=\sin(\pi x)$다. 양자 PDE 방법을 평가하기 전에 discretization error와 condition number를 측정한다.

In [ ]:
def solve_poisson(interior_points):
    h = 1.0 / (interior_points + 1)
    x = np.arange(1, interior_points + 1) * h
    matrix = (2 * np.eye(interior_points) - np.eye(interior_points, k=1) - np.eye(interior_points, k=-1)) / h**2
    rhs = np.pi**2 * np.sin(np.pi * x)
    numeric = np.linalg.solve(matrix, rhs)
    exact = np.sin(np.pi * x)
    return {
        'n': interior_points,
        'h': h,
        'condition': float(np.linalg.cond(matrix)),
        'l2_error': float(np.sqrt(h) * np.linalg.norm(numeric - exact)),
        'linf_error': float(np.max(np.abs(numeric - exact))),
    }

poisson_results = [solve_poisson(n) for n in (15, 31, 63)]
for result in poisson_results:
    print(result)

assert poisson_results[-1]['linf_error'] < poisson_results[0]['linf_error']
assert poisson_results[-1]['condition'] > poisson_results[0]['condition']

grid를 세분화하면 discretization error는 줄지만 선형계의 condition number는 커진다. HHL/QLSA 분석에서는 이 `κ` 의존성과 원하는 출력 정밀도를 반드시 복잡도에 포함한다.

## 4. HHL/QLSA 적용 전 타당성 감사

아래 코드는 HHL을 구현하지 않는다. 같은 선형계에 대해 어떤 출력이 필요한지와 숨은 비용을 드러내는 감사 양식이다.

In [ ]:
def linear_system_audit(matrix, rhs, output_contract):
    solution = np.linalg.solve(matrix, rhs)
    normalized_state = solution / np.linalg.norm(solution)
    return {
        'dimension': matrix.shape[0],
        'sparsity_per_row_max': int(np.max(np.count_nonzero(matrix, axis=1))),
        'condition_number': float(np.linalg.cond(matrix)),
        'rhs_norm': float(np.linalg.norm(rhs)),
        'output_contract': output_contract,
        'normalized_solution_state': normalized_state,
        'must_cost_state_preparation': True,
        'must_cost_readout': True,
    }

well_conditioned = np.array([[2.0, -0.2], [-0.2, 1.0]])
ill_conditioned = np.diag([1.0, 1e-4])
rhs = np.array([1.0, 1.0])
audits = [
    linear_system_audit(well_conditioned, rhs, '해 상태에서 특정 observable'),
    linear_system_audit(ill_conditioned, rhs, '전체 고전 해 벡터'),
]
for audit in audits:
    print({key: value for key, value in audit.items() if key != 'normalized_solution_state'})

assert audits[1]['condition_number'] > 1_000 * audits[0]['condition_number']

## 결과를 해석할 때 반드시 답할 질문

- `|b>`와 matrix oracle/block encoding을 만드는 비용은 얼마인가?
- sparsity와 condition number 가정이 실제 문제에서 유지되는가?
- 필요한 출력은 전체 vector인가, 해 상태의 소수 observable인가?
- precision, success probability, shots와 postselection 비용을 넣었는가?
- direct/sparse/Krylov/spectral 같은 강한 고전 solver와 같은 계약으로 비교했는가?
- NISQ variational 실험과 fault-tolerant QLSA estimate를 섞지 않았는가?

### 확장 과제

1. VQE energy를 finite shots로 추정해 seed별 confidence interval을 만든다.
2. QAOA에 shot sampling을 추가하고 기대값·best sample·success probability를 분리한다.
3. Poisson grid별 관측량 하나와 전체 field 복원 비용을 비교한다.
4. Qiskit/PennyLane 공식 예제로 옮기되 version, backend, depth와 shots를 기록한다.